# 02 · Test Alpha Vantage as the second transcript source

**Decision 2, Path 2:** Option A (`kurry/sp500_earnings_transcripts`) misses decliners after they leave the S&P 500, Gap, JCP and small caps.
Before paying $99 for API Ninjas Business, test whether Alpha Vantage's **free** tier fills those gaps.

**4 test calls** (free tier ≈ 25 requests/day, 5/minute):

| # | Call | What it tests |
|---|---|---|
| 1 | BBBY 2021Q2 | decliner after leaving the index |
| 2 | GPS 2019Q2 | company missing from Option A |
| 3 | BGFV 2017Q2 | small-cap coverage |
| 4 | M 2018Q1 | overlap with Option A → compare the text |

**Rules:** every response is saved to `data/raw/alphavantage/` and never requested twice; the API key lives in `.env`, never in this notebook.

In [8]:
import json
import os
import time
from pathlib import Path

import requests
from dotenv import find_dotenv, load_dotenv

# find_dotenv walks UP from the current folder until it finds .env.
# usecwd=True matters in Jupyter: without it, find_dotenv looks relative to the
# library's own file instead of where the notebook is running.
ENV_PATH = find_dotenv(usecwd=True)
assert ENV_PATH, ".env not found - create it in the project root"
load_dotenv(ENV_PATH)

PROJECT_ROOT = Path(ENV_PATH).parent  # .env sits in the project root, so this anchors every other path
API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

# never print the key itself - a printed key ends up in the saved notebook output (and on GitHub)
assert API_KEY, "ALPHAVANTAGE_API_KEY missing from .env"
print("project root:", PROJECT_ROOT)
print("API key loaded:", API_KEY[:2] + "...")

project root: /Users/vrushankhitenrawal/Desktop/spin-detector
API key loaded: X8...


In [9]:
BASE_URL = "https://www.alphavantage.co/query"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "alphavantage"
RAW_DIR.mkdir(parents=True, exist_ok=True)  # parents=True creates data/ and data/raw/ too

# (symbol, quarter, what the call tests)
TEST_CALLS = [
    ("BBBY", "2021Q2", "decliner after leaving the index"),
    ("GPS",  "2019Q2", "missing from Option A"),
    ("BGFV", "2017Q2", "small-cap coverage"),
    ("M",    "2018Q1", "overlap with Option A"),
    # GPS came back empty. Gap changed its ticker GPS -> GAP in 2024: does Alpha
    # Vantage file the company's whole history under today's ticker?
    ("GAP",  "2019Q2", "ticker change: Gap's history under its new ticker?"),
]

# Alpha Vantage returns HTTP 200 even for errors, with one of these keys in the JSON body
ERROR_KEYS = ("Note", "Information", "Error Message")

In [10]:
def fetch_transcript(symbol, quarter):
    """Return the transcript JSON for one call, from the local cache if we already have it.

    Returns (data, from_cache). Raises RuntimeError on an API-level error so a bad
    response is never saved to disk (otherwise the cache would serve the error forever).
    """
    path = RAW_DIR / f"{symbol}_{quarter}.json"

    # 1. cache first: a request we've already paid for is never made again
    if path.exists():
        return json.loads(path.read_text()), True

    # 2. otherwise hit the API
    params = {
        "function": "EARNINGS_CALL_TRANSCRIPT",
        "symbol": symbol,
        "quarter": quarter,
        "apikey": API_KEY,
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)  # timeout: never hang forever
    except requests.RequestException as err:
        # network errors (timeout, no connection) also print the URL with the key,
        # so report only the error type; "from None" hides the original traceback
        raise RuntimeError(f"{symbol} {quarter}: network error ({type(err).__name__})") from None

    # NOT response.raise_for_status(): its error message contains the full URL,
    # which includes ?apikey=... and would leak the key into the notebook output
    if not response.ok:
        raise RuntimeError(f"{symbol} {quarter}: HTTP {response.status_code}")
    data = response.json()

    # 3. catch the "HTTP 200 but actually an error" case
    for key in ERROR_KEYS:
        if key in data:
            raise RuntimeError(f"{symbol} {quarter}: {key} -> {data[key]}")

    # 4. only a good response gets saved
    path.write_text(json.dumps(data, indent=2))
    return data, False

### Run the test calls
The first run spent **4 of the 25 daily requests**. Re-running is free for those 4: they come from `data/raw/alphavantage/`. Only the new `GAP 2019Q2` call hits the API (**1 request**).
It waits 15 seconds between real requests to stay under the 5-requests-per-minute limit.

In [11]:
results = {}  # (symbol, quarter) -> JSON, only for calls that succeeded

for i, (symbol, quarter, purpose) in enumerate(TEST_CALLS):
    try:
        data, from_cache = fetch_transcript(symbol, quarter)
        results[(symbol, quarter)] = data
        print(f"OK    {symbol} {quarter} ({'cache' if from_cache else 'API'}) - {purpose}")
    except RuntimeError as err:
        from_cache = False  # a failed call still used a request
        print(f"FAIL  {err}")

    # pause only after a REAL request, and not after the last one
    if not from_cache and i < len(TEST_CALLS) - 1:
        time.sleep(15)

OK    BBBY 2021Q2 (cache) - decliner after leaving the index
OK    GPS 2019Q2 (cache) - missing from Option A
OK    BGFV 2017Q2 (cache) - small-cap coverage
OK    M 2018Q1 (cache) - overlap with Option A
OK    GAP 2019Q2 (API) - ticker change: Gap's history under its new ticker?


### Inspect each response
The exact response shape isn't documented clearly, so look before assuming: print the top-level keys and the first few segments.

In [12]:
def describe(symbol, quarter, data):
    segments = data.get("transcript") or []  # "or []" also covers a key that exists but is None
    print(f"=== {symbol} {quarter} ===")
    print("top-level keys:", list(data.keys()))
    if not segments:
        print("NO TRANSCRIPT returned\n")
        return
    print("segments:", len(segments), "| keys per segment:", list(segments[0].keys()))
    for seg in segments[:3]:
        print(f"  - {seg.get('speaker')} ({seg.get('title')}): {str(seg.get('content'))[:120]}...")
    print()


for (symbol, quarter), data in results.items():
    describe(symbol, quarter, data)

=== BBBY 2021Q2 ===
top-level keys: ['symbol', 'quarter', 'transcript']
segments: 74 | keys per segment: ['speaker', 'title', 'content', 'sentiment']
  - Operator (Operator): Welcome to Bed Bath & Beyond's Fiscal 2021 Second Quarter Earnings Conference Call. My name is John. I will be your oper...
  - Susie Kim (VP of Investor Relations): Thank you, and good morning, everyone. Welcome to our fiscal 2021 second quarter earnings call. Joining us today are Mar...
  - Mark Tritton (President and CEO): Thank you, Susie, and good morning, everyone. After several consecutive quarters of delivering positively on our plan, t...

=== GPS 2019Q2 ===
top-level keys: ['symbol', 'quarter', 'transcript']
NO TRANSCRIPT returned

=== BGFV 2017Q2 ===
top-level keys: ['symbol', 'quarter', 'transcript']
segments: 33 | keys per segment: ['speaker', 'title', 'content', 'sentiment']
  - Operator (Operator): Good day, ladies and gentlemen, and welcome to the Big 5 Sporting Goods Second Quarter 2017 Earnings R

### Summary table
- **split_by_speaker:** more than one distinct speaker means the text comes pre-split by speaker turn (what Decision 6 needs for chunking)
- **analysts_present:** analysts only speak during Q&A, so an analyst title means the Q&A section is included

In [13]:
import pandas as pd

rows = []
for symbol, quarter, purpose in TEST_CALLS:
    data = results.get((symbol, quarter)) or {}
    segments = data.get("transcript") or []
    titles = [str(seg.get("title", "")).lower() for seg in segments]
    speakers = {seg.get("speaker") for seg in segments}
    rows.append({
        "call": f"{symbol} {quarter}",
        "tests": purpose,
        "returned": bool(segments),
        "segments": len(segments),
        "speakers": len(speakers),
        "split_by_speaker": len(speakers) > 1,
        "analysts_present": any("analyst" in t for t in titles),
        "words": sum(len(str(seg.get("content", "")).split()) for seg in segments),
    })

summary = pd.DataFrame(rows)
summary

,call,tests,returned,segments,speakers,split_by_speaker,analysts_present,words
0,BBBY 2021Q2,decliner after leaving the index,True,74,16,True,True,7444
1,GPS 2019Q2,missing from Option A,False,0,0,False,False,0
2,BGFV 2017Q2,small-cap coverage,True,33,5,True,True,3168
3,M 2018Q1,overlap with Option A,True,79,16,True,True,9052
4,GAP 2019Q2,ticker change: Gap's history under its new tic...,True,61,15,True,True,9677


### Overlap check: Macy's 2018Q1, Alpha Vantage vs Option A
Two questions in one cell:
1. **Do the two sources contain the same call?** Compare the Alpha Vantage text with every 2018 Macy's transcript in Option A. Word-overlap (Jaccard) similarity: ~0.6+ means the same call, ~0.2 means different calls.
2. **Is Alpha Vantage's `quarter` fiscal or calendar?** The best-matching Option A row tells us. Option A's `year`/`quarter` are fiscal (Notebook 01 finding), and Macy's fiscal Q1 2018 call was in **May 2018**. If the best match is the May call, Alpha Vantage's `2018Q1` means fiscal Q1 too.

In [14]:
from datasets import load_dataset

av_macy = results.get(("M", "2018Q1"))
if not av_macy or not av_macy.get("transcript"):
    print("No Alpha Vantage transcript for M 2018Q1 - nothing to compare")
else:
    av_text = " ".join(str(seg.get("content", "")) for seg in av_macy["transcript"])

    ds = load_dataset("kurry/sp500_earnings_transcripts", split="train")  # already cached from Notebook 01
    macy_2018 = ds.filter(
        lambda b: [s == "M" and d.startswith("2018") for s, d in zip(b["symbol"], b["date"])],
        batched=True,
    )

    def jaccard(a, b):
        # share of distinct words the two texts have in common: 1.0 = identical vocabulary
        words_a, words_b = set(a.lower().split()), set(b.lower().split())
        return len(words_a & words_b) / max(1, len(words_a | words_b))

    for row in macy_2018:
        score = jaccard(av_text, row["content"])
        print(f"{row['date']}  labelled FY{row['year']} Q{row['quarter']}  similarity {score:.2f}")

2018-02-27 10:00:00  labelled FY2017 Q4  similarity 0.33
2018-11-14 09:30:00  labelled FY2018 Q3  similarity 0.31
2018-08-15 09:30:00  labelled FY2018 Q2  similarity 0.33
2018-05-16 09:30:00  labelled FY2018 Q1  similarity 0.80


### Record the results (filled 2026-10-20)
| Call | Returned? | Split by speaker? | Q&A included? | Notes |
|---|---|---|---|---|
| BBBY 2021Q2 | ✅ 74 segments, 7,444 words | ✅ 16 speakers, with titles | ✅ analysts present | **Decliner after leaving the S&P 500 is covered**. Option A stops at Apr 2018. Operator says "Fiscal 2021 Second Quarter" → `quarter` is fiscal |
| GPS 2019Q2 | ❌ empty (`transcript: []`) | n/a | n/a | Not a coverage gap: wrong ticker (see GAP row) |
| BGFV 2017Q2 | ✅ 33 segments, 3,168 words | ✅ 5 speakers | ✅ analysts present | **Small caps are covered** |
| M 2018Q1 | ✅ 79 segments, 9,052 words | ✅ 16 speakers | ✅ analysts present | Matches Option A's **May 2018 (FY2018 Q1)** call: Jaccard **0.80** vs 0.31–0.33 for the other 2018 calls → same call, and `quarter` = **fiscal** |
| GAP 2019Q2 | ✅ 61 segments, 9,677 words | ✅ 15 speakers | ✅ analysts present | Same call that was empty under GPS → Alpha Vantage files history under **today's ticker** → need a ticker-history map keyed on CIK |

**Requests used:** 5 of 25 (free tier, 2026-10-20).

**Conclusion:** Alpha Vantage's free tier passes the decision rule. It covers a decliner after index exit (BBBY), a company missing from Option A (Gap, under its current ticker) and a small cap (BGFV), all speaker-split with Q&A, using fiscal quarters that join cleanly with Option A. **API Ninjas Business ($99) is not needed.**

**Design consequences:**
1. Identify companies by **SEC CIK**, never by ticker; request transcripts by the company's *current* ticker via a ticker-history map (Decision 5).
2. Sanity-check every transcript: the company name must appear in the operator's opening line, which catches a reused ticker pulling in another company's calls.
3. **Never use Alpha Vantage's `sentiment` field.** It's third-party scoring with a hindsight-leakage risk, and narrative scoring is this project's own job.
4. Transcripts from the two sources aren't word-identical (0.80, not 1.0) → if mixing sources, keep a `source` column and check backtest performance per source (confounding rule).

**Still open:** A (hybrid, free, ~6 weeks of downloads) vs B (Alpha Vantage single source, one premium month).